# 02 - Evaluation Dataset

This notebook demonstrates the deterministic normalization of the versioned evaluation CSV files into aligned MCQ and no-hint JSONL datasets.

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from legal_rag.evaluation_dataset import EvaluationDatasetConfig, run_evaluation_dataset

config = EvaluationDatasetConfig(
    mcq_source=str(ROOT / "data/evaluation/questions.csv"),
    no_hint_source=str(ROOT / "data/evaluation/questions_no_hint.csv"),
    output_dir=str(ROOT / "data/evaluation_clean"),
)
manifest = run_evaluation_dataset(config)
print(json.dumps({"ready_for_evaluation": manifest["ready_for_evaluation"], "counts": manifest["counts"]}, ensure_ascii=False, indent=2))

{
  "ready_for_evaluation": true,
  "counts": {
    "mcq": 100,
    "no_hint": 100,
    "mcq_source_rows": 163,
    "no_hint_source_rows": 100,
    "mcq_dropped_empty_rows": 63,
    "no_hint_dropped_empty_rows": 0
  }
}


The no-hint dataset removes answer choices while preserving the same question intent, answer text, level, and legal references. This enables fair comparison between no-RAG, simple RAG, and advanced RAG settings.

In [9]:
output_dir = Path(config.output_dir)
profile = json.loads((output_dir / "evaluation_profile.json").read_text(encoding="utf-8"))
quality = (output_dir / "quality_report.md").read_text(encoding="utf-8")

print("Level distribution:")
print(json.dumps(profile["level_distribution"], ensure_ascii=False, indent=2))
print("\nSample references:")
for ref in profile["sample_references"][:5]:
    print("-", ref)

Level distribution:
{
  "L1": 25,
  "L2": 25,
  "L3": 25,
  "L4": 25
}

Sample references:
- Legge regionale 1 agosto 2022, n. 19 - Art. 2
- Legge regionale 1 agosto 2022, n. 19 - Art. 4
- Legge regionale 1 agosto 2022, n. 19 - Art. 6
- Legge regionale 1 agosto 2022, n. 19 - Art. 9
- Legge regionale 10 giugno 1983, n. 56 - Art. 1


In [10]:
print("Alignment examples:")
for item in profile["alignment_examples"]:
    print(json.dumps(item, ensure_ascii=False, indent=2))

Alignment examples:
{
  "correct_answer": "Il direttore generale il collegio sindacale",
  "mcq_stem": "Quali sono gli organi dell'azienda Usl?",
  "no_hint_question": "Quali sono gli organi dell'azienda Usl?",
  "qid": "eval-0001"
}
{
  "correct_answer": "Attraverso il Piano socio-sanitario regionale",
  "mcq_stem": "Come si attua la programmazione sanitaria regionale?",
  "no_hint_question": "Come si attua la programmazione sanitaria regionale?",
  "qid": "eval-0002"
}
{
  "correct_answer": "Le relative funzioni sono svolte dal direttore amministrativo",
  "mcq_stem": "Cosa accade in caso di vacanza dell'ufficio o assenza o impedimento del direttore generale, qualora non ricorrano gravi motivi?",
  "no_hint_question": "Cosa accade in caso di vacanza dell'ufficio o assenza o impedimento del direttore generale, qualora non ricorrano gravi motivi?",
  "qid": "eval-0003"
}


In [11]:
print(quality)

# 02 - Evaluation Dataset Quality Report

- Ready for evaluation: **True**
- Generated at UTC: `2026-05-02T17:29:51Z`

## Quality Gates
- `source_files_readable`: **True**
- `expected_record_count`: **True**
- `record_counts_match`: **True**
- `required_fields_present`: **True**
- `required_fields_non_empty`: **True**
- `stable_qids_aligned`: **True**
- `level_distribution_reported`: **True**
- `source_hashes_recorded`: **True**
- `output_files_exist_and_hash`: **True**

## Counts
- MCQ records: 100
- No-hint records: 100
- MCQ source rows: 163
- No-hint source rows: 100
- MCQ dropped empty rows: 63
- No-hint dropped empty rows: 0

## Level Distribution
- L1: 25
- L2: 25
- L3: 25
- L4: 25

